# Sentimiento de la hinchada (NLP)

Objetivo: medir el hype/estado de ánimo de la hinchada xeneize a partir de
Reddit y combinarlo con el ranking de candidatos para el tweet semanal.

## Limitaciones actuales (documentadas)
- VADER está entrenado en inglés: se aplica con un **lexicón mínimo de
  fútbol/rioplatense** (parche best-effort, no un modelo de español real).
- Las credenciales de Reddit (tipo de app) deben permitir acceso read-only
  tipo *script*. Si la app es "web/installed" o las credenciales fallan, el
  pipeline **no se rompe**: cae a *placeholders* sintéticos claramente
  marcados (`origen='placeholder'`) para que la automatización siga rodando.
- Subreddits objetivo configurables en `SUBS`. Verificar el nombre exacto
  cuando la API esté disponible (r/BocaJuniors y r/xeneizes son candidatos).


In [ ]:
import os
import sys
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
load_dotenv('../secrets/.env')

sns.set_style('whitegrid')
OUTPUTS = '../outputs'
DATA = '../data'
os.makedirs(OUTPUTS, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

## Lexicón mínimo español-futbolero

Parche sobre VADER: palabras locales con polaridad. Se documenta que no
sustituye a un modelo de sentimiento en español (deuda técnica conocida).

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

LEXICO_ESPANIOL = {
    'boquita': 1.5, 'xeneize': 1.0, 'glorioso': 1.2, 'ganamos': 2.5,
    'goleamos': 3.0, 'despedazar': -2.5, 'papelon': -2.8, 'vergüenza': -2.0,
    'pases': 0.4, 'triunfo': 2.2, 'puto': -1.5, 'fiasco': -2.3,
    'no juega': -1.5, 'fracaso': -2.6,
}

analyzer = SentimentIntensityAnalyzer()
analyzer.lexicon.update(LEXICO_ESPANIOL)

## Extracción desde Reddit

Intenta leer publicaciones de los subs configurados. Si el login read-only
falla (401 o similar), devuelve `None` para que el notebook caiga al
placeholder sin errores.

In [ ]:
SUBS = ['BocaJuniors', 'xeneizes']
POSTS_LIMITE = 40
COMENTARIOS_POR_POST = 20


def extraer_reddit():
    import praw
    reddit = praw.Reddit(
        client_id=os.getenv('REDDIT_CLIENT_ID'),
        client_secret=os.getenv('REDDIT_CLIENT_SECRET'),
        user_agent='boca-scouting-ds/0.1 (investigacion)',
    )
    textos = []
    for sub_name in SUBS:
        sub = reddit.subreddit(sub_name)
        for post in sub.hot(limit=POSTS_LIMITE):
            textos.append((post.title, datetime.fromtimestamp(post.created_utc)))
            for comentario in list(post.comments[:COMENTARIOS_POR_POST]):
                body = getattr(comentario, 'body', None)
                if body:
                    textos.append((body, datetime.fromtimestamp(comentario.created_utc)))
    return textos if textos else None

In [ ]:
def cargar_placeholders():
    plantillas = [
        'Que grande Boca, este pibe rinde y encima a coste bajo',
        'No lo traigan por favor, no rinde ni en el barrio',
        'Con boquita la rompemos siempre, ficharlo ya',
        'Otro refuerzo que no le da al nivel, olvidate',
        'Juega bien pero es carisimo, mejor mirar al mercado',
        'Este tiene ADN Boca de verdad, ojalá venga',
        'Un papelón en la última fecha, no necesitamos mas refuerzos así',
        'El club tiene que salir a buscar cracks, no descartados',
        'Goleamos y encima aparece este nombre, buen momento',
        'No conozco al pibe, alguno lo vio jugar?',
        'Ficharlo es un fiasco, mejor no gastar plata',
        'Ese es un jugador de selección, fuera de serie',
    ]
    ahora = datetime.now()
    filas = []
    for i, t in enumerate(plantillas * 3):
        fecha = ahora - timedelta(days=13, hours=i * 9)
        filas.append({'texto': t, 'fecha': fecha, 'origen': 'placeholder'})
    return filas

In [ ]:
if os.getenv('REDDIT_CLIENT_ID'):
    try:
        crudos = extraer_reddit()
    except Exception as e:
        print(f'[aviso] Reddit no disponible ({type(e).__name__}): {e}')
        crudos = None
else:
    print('[aviso] REDDIT_CLIENT_ID vacio: se usan placeholders')
    crudos = None

if crudos:
    df = pd.DataFrame(crudos, columns=['texto', 'fecha'])
    df['origen'] = 'reddit'
    print(f'Reddit OK: {len(df)} textos')
else:
    df = pd.DataFrame(cargar_placeholders())
    print(f'PLACEHOLDER (sintetico): {len(df)} textos')

In [ ]:
res = df['texto'].apply(analyzer.polarity_scores).apply(pd.Series)
df = pd.concat([df, res], axis=1)

def clasificar(compound):
    if compound >= 0.05:
        return 'positivo'
    if compound <= -0.05:
        return 'negativo'
    return 'neutral'

df['clasificacion'] = df['compound'].apply(clasificar)
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha')

print(df['clasificacion'].value_counts().to_string())

## Hype score semanal

Se agrupa por semana (lunes a domingo) y se calcula:

- `net_sentiment`: promedio del compound (rango [-1, 1]).
- `positividad`: fracción de textos positivos.
- `volumen`: cantidad de textos.
- `hype`: `(0.6*net_sentiment + 0.4*positividad)` atenuado por volumen
  (`min(1, log1p(volumen)/3)`) para que una semana con 2 comentarios no
  infle el score.

In [ ]:
semana = df['fecha'].dt.to_period('W')
resumen = df.groupby(semana).agg(
    net_sentiment=('compound', 'mean'),
    positividad=('clasificacion', lambda s: (s == 'positivo').mean()),
    volumen=('texto', 'count'),
).rename_axis('semana').reset_index()
resumen['fecha'] = resumen['semana'].dt.to_timestamp()

vol_atenuacion = np.minimum(1.0, np.log1p(resumen['volumen']) / 3.0)
resumen['hype'] = (0.6 * resumen['net_sentiment'] + 0.4 * resumen['positividad']) * vol_atenuacion
resumen = resumen.round(4)
print(resumen.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.countplot(data=df, x='clasificacion', order=['positivo', 'neutral', 'negativo'], ax=axes[0])
axes[0].set_title('Distribución de sentimiento por comentario')
axes[1].plot(resumen['fecha'], resumen['hype'], marker='o')
axes[1].axhline(0, color='gray', ls='--', lw=0.8)
axes[1].set_title('Hype score semanal de la hinchada')
axes[1].set_ylabel('hype')
plt.tight_layout()
plt.savefig(f'{OUTPUTS}/16_sentimiento_hinchada.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df.to_csv(f'{DATA}/sentimiento_hinchada.csv', index=False, encoding='utf-8-sig')
print(f'Guardado data/sentimiento_hinchada.csv ({len(df)} filas)')

## Interpretación
- `hype > 0`: clima positivo (buen momento para anunciar fichajes).
- `hype < 0`: clima negativo (la hinchada presiona; el tweet debe ser cauto).
- Con `origen='placeholder'` los números son sintéticos y no reflejan a la
  hinchada real: revisar credenciales Reddit (app tipo *script*) para
  activar el scrape real sin cambiar código.